# Summer Experiments Aich HI 

In [4]:
# testing hysteresis parameter calculation? 
import numpy as np
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

experiment_directory = 'pulse_experiments'
experiments = {}
for filename in os.listdir(experiment_directory):
    # check if the file is a CSV file
    if filename.endswith('.csv'):
        file_path = os.path.join(experiment_directory, filename) # construct the full file path
        df = pd.read_csv(file_path)                         # read the CSV file into a data frame
        df = df.dropna(subset=['Date_Time'])                # drop rows where 'Date/Time' is NaN  
        df['Date_Time'] = pd.to_datetime(df['Date_Time'])   # convert to datetime format
        df = df.set_index('Date_Time')                      # set date time as the index 
        df = df.dropna(how='all', axis=1)                   # drop columns where all values are NaN
        df = df.loc[:, ~df.columns.astype(str).str.startswith('Unnamed')]  # drop stray unnamed columns
        key = filename[:-4]                                 # remove the '.csv' from the filename to use as the dictionary key
        experiments[key] = df                                      # store the data frame in the dictionary

# calculate a shear stress column for each experiment
for experiment_name, experiment_df in experiments.items():
    # calculate shear stress using the formula: shear_stress = density * gravity * water_depth * slope
    density = 1000  # kg/m^3
    gravity = 9.81  # m/s^2
    slope = 0.098
    water_depth = experiment_df['Depth']  # in cm
    shear_stress = density * gravity * water_depth/1000 * slope
    experiments[experiment_name]['shear_stress'] = shear_stress

In [5]:
experiments['Experiment2']

,Lab ID,Depth,SSC (uL/L),SSC (mg/L),DOC (mg/L),POC (mg/L),PP (mg/L),SRP (mg/L),TP (mg/L),shear_stress
Date_Time,,,,,,,,,,
2022-08-02 11:57:00,1,9.275,97.69,167.044168,2.640,10.974524,0.010039,0.024571,0.060181,8.916800
2022-08-02 11:57:00,2,22.750,347.61,145.078620,2.384,12.330497,0.022338,0.017498,0.036531,21.871395
2022-08-02 11:57:00,3,19.250,191.85,76.137885,2.284,NaN,0.009810,0.021035,0.032893,18.506565
2022-08-02 11:57:00,4,16.100,176.97,47.967685,2.423,4.967149,0.010319,0.029876,0.032893,15.478218
2022-08-02 11:57:00,5,13.884,127.27,67.326074,2.488,8.890353,0.006210,0.026340,0.034712,13.347800
2022-08-02 11:58:00,6,12.775,83.04,61.268422,2.473,NaN,0.005243,0.028108,0.089288,12.281630
2022-08-02 11:58:00,7,12.134,68.69,40.782355,2.314,NaN,0.003300,0.028108,0.038350,11.665385
2022-08-02 11:58:00,8,11.959,62.04,43.350237,2.402,NaN,0.002601,0.029876,0.034712,11.497143
2022-08-02 11:58:00,9,11.959,61.07,37.341299,2.278,NaN,0.002619,0.024571,0.040170,11.497143


Hysteresis index calculation functions

In [6]:
def normalize_data(df, tau_col, constituent_col):
    pair = df[[tau_col, constituent_col]].dropna().sort_index()

    if pair.empty:
        return None

    pair = pair.copy()
    
    pair[tau_col] = pd.to_numeric(pair[tau_col], errors='coerce')
    pair[constituent_col] = pd.to_numeric(pair[constituent_col],errors='coerce')
    pair = pair.dropna(subset=[tau_col, constituent_col])

    tau_max = pair[tau_col].max()
    cn_max = pair[constituent_col].max()

    if pd.isna(tau_max) or pd.isna(cn_max) or tau_max == 0 or cn_max == 0:
        return None

    pair["tau_n"] = pair[tau_col] / tau_max
    pair["cn"] = pair[constituent_col] / cn_max
    return pair

def split_hydrograph(norm_df):
    if norm_df.empty:
        return None, None, None

    peak_pos = norm_df["tau_n"].to_numpy().argmax()

    rise = norm_df.iloc[:peak_pos + 1][["tau_n", "cn"]]
    fall = norm_df.iloc[peak_pos + 1:][["tau_n", "cn"]]

    return rise, fall, peak_pos


def reference_line(norm_df, peak_pos):
    peak_row = norm_df.iloc[peak_pos]
    last_row = norm_df.iloc[-1]

    x1 = peak_row["tau_n"]
    y1 = peak_row["cn"]
    x2 = last_row["tau_n"]
    y2 = last_row["cn"]

    A = y1 - y2
    B = x2 - x1
    C = x1 * y2 - x2 * y1

    return A, B, C


def distance_from_line(df_xy, A, B, C):
    return (A * df_xy["tau_n"] + B * df_xy["cn"] + C) / np.sqrt(A**2 + B**2)


def projection_to_line(x0, y0, A, B, C):
    d = (A * x0 + B * y0 + C) / (A**2 + B**2)
    x_proj = x0 - A * d
    y_proj = y0 - B * d
    return x_proj, y_proj


def calculate_aich_HI(df, tau_col, constituent_col, storm_name):
    norm_df = normalize_data(df, tau_col, constituent_col)

    if norm_df is None or len(norm_df) < 3:
        print(f"Not enough paired data to calculate AICH HI for storm {storm_name}")
        return None

    rise, fall, peak_pos = split_hydrograph(norm_df)

    if rise is None or fall is None or rise.empty or fall.empty:
        print(f"Not enough data points to calculate AICH HI for storm {storm_name}")
        return None

    A, B, C = reference_line(norm_df, peak_pos)

    rise_dist = -distance_from_line(rise, A, B, C)
    fall_dist = distance_from_line(fall, A, B, C)

    if rise_dist.empty or fall_dist.empty:
        print(f"Not enough data points to calculate AICH HI for storm {storm_name}")
        return None

    rise_pos = rise_dist.abs().to_numpy().argmax()
    fall_pos = fall_dist.abs().to_numpy().argmax()

    Drise = rise_dist.iloc[rise_pos]
    Dfall = fall_dist.iloc[fall_pos]
    HI = Drise + Dfall

    rise_point = rise.iloc[rise_pos]
    fall_point = fall.iloc[fall_pos]

    results = {
        "HI": HI,
        "Drise": Drise,
        "Dfall": Dfall,
        "storm_name": storm_name,
        "peak tau": norm_df.iloc[peak_pos]["tau_n"],
        "peak constituent": norm_df.iloc[peak_pos]["cn"],
        "rise": rise,
        "fall": fall,
        "A": A,
        "B": B,
        "C": C,
        "Drise_point": (rise_point["tau_n"], rise_point["cn"]),
        "Dfall_point": (fall_point["tau_n"], fall_point["cn"]),
        "norm_df": norm_df,
    }

    return results




def plot_aich_hysteresis(results, df, tau_col, constituent_col,
                        xlabel=r"Normalized Shear Stress ($\tau_n$)",
                        ylabel="Normalized Constituent",
                        out_dir="HI_calculations/aich_plots",
                        save=True,
                        show=False):
    
    norm_df = results["norm_df"]
    rise = results["rise"]
    fall = results["fall"]
    A = results["A"]
    B = results["B"]
    C = results["C"]
    Drise = results["Drise"]
    Dfall = results["Dfall"]
    HI = results["HI"]
    storm_name = results.get("storm_name", "")

    xr, yr = results["Drise_point"]
    xf, yf = results["Dfall_point"]

    xr_proj, yr_proj = projection_to_line(xr, yr, A, B, C)
    xf_proj, yf_proj = projection_to_line(xf, yf, A, B, C)


    tau_n_vals = norm_df["tau_n"].values
    xmin = min(tau_n_vals.min(), float(xr_proj), float(xf_proj)) - 0.05
    xmax = max(tau_n_vals.max(), float(xr_proj), float(xf_proj)) + 0.05

    xline = np.linspace(xmin, xmax, 200)
    yline = -(A * xline + C) / B

    ts_df = df[[tau_col, constituent_col]].dropna().sort_index()

    fig, axes = plt.subplots(1, 2, figsize=(10, 5), gridspec_kw={"width_ratios": [1.2, 1]})
    # left: time series
    ax = axes[0]
    ax.plot(ts_df.index, ts_df[tau_col], color="tab:blue", linewidth=1.5, label=tau_col)
    ax.set_ylabel(tau_col, color="tab:blue")
    ax.tick_params(axis="y", labelcolor="tab:blue")
    ax.xaxis.set_major_locator(MaxNLocator(8))

    ax2 = ax.twinx()
    ax2.plot(ts_df.index, ts_df[constituent_col], color="tab:red", linewidth=1.5, label=constituent_col)
    ax2.set_ylabel(constituent_col, color="tab:red")
    ax2.tick_params(axis="y", labelcolor="tab:red")

    ax.set_title("Event time series")
    ax.set_xlabel("Time")
    ax.grid(True, alpha=0.3)

    # Right: hysteresis pattern
    ax = axes[1]
    ax.plot(norm_df["tau_n"], norm_df["cn"], color="black", linewidth=2, alpha=0.7, zorder=1)
    ax.scatter(rise["tau_n"], rise["cn"], color="forestgreen", linewidth=2, label="Rising limb", zorder=4)
    ax.scatter(fall["tau_n"], fall["cn"], color="firebrick", linewidth=2, label="Falling limb", zorder=4)
    ax.plot(xline, yline, "--", color="gray", linewidth=2, label="Reference line", zorder=2)

    ax.scatter(xr, yr, s=80, color="forestgreen", zorder=5)
    ax.plot([xr, xr_proj], [yr, yr_proj], "--", color="forestgreen", linewidth=2, zorder=3)
    ax.scatter(xf, yf, s=80, color="firebrick", zorder=5)
    ax.plot([xf, xf_proj], [yf, yf_proj], "--", color="firebrick", linewidth=2, zorder=3)

    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)

    # make axes limits according to the data range, but with some padding
    geom_x = np.r_[norm_df["tau_n"].to_numpy(), xr, xr_proj, xf, xf_proj]
    geom_y = np.r_[norm_df["cn"].to_numpy(), yr, yr_proj, yf, yf_proj]
    x_min, x_max = geom_x.min(), geom_x.max()
    y_min, y_max = geom_y.min(), geom_y.max()
    x_span = x_max - x_min
    y_span = y_max - y_min
    span = max(x_span, y_span)
    x_mid = 0.5 * (x_min + x_max)
    y_mid = 0.5 * (y_min + y_max)
    pad = 0.05 * span
    ax.set_xlim(x_mid - span / 2 - pad, x_mid + span / 2 + pad)
    ax.set_ylim(y_mid - span / 2 - pad, y_mid + span / 2 + pad)
    ax.set_aspect("equal", adjustable="box")

    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.set_title(f"Drise = {Drise:.2f}, Dfall = {Dfall:.2f}, HI = {HI:.2f}")

    main_title = f"{storm_name} - {constituent_col} Hysteresis" if storm_name else f"{constituent_col} Hysteresis"
    fig.suptitle(main_title, fontsize=15)
    fig.tight_layout(rect=[0, 0, 1, 0.95])

    if save:
        os.makedirs(out_dir, exist_ok=True)
        safe_name = f"{storm_name}_{constituent_col}_aich.png".replace(" ", "_").replace("/", "_")
        fig.savefig(os.path.join(out_dir, safe_name), dpi=300, bbox_inches="tight")

    if show:
        plt.show()

    plt.close(fig)

Calculate constituents hysteresis 

In [7]:
all_results = []

for experiment_name, experiment_df in experiments.items():
    for constituent in ["SSC (mg/L)", "DOC (mg/L)", "POC (mg/L)", "PP (mg/L)", "SRP (mg/L)", "TP (mg/L)"]:
        if constituent not in experiment_df.columns:
            continue
        result = calculate_aich_HI(
            experiment_df,
            tau_col="Depth",
            constituent_col=constituent,
            storm_name=experiment_name)

        if result is not None:
            result["storm"] = experiment_name
            result["constituent"] = constituent
            all_results.append(result)

all_results = pd.DataFrame(all_results)
all_results.to_csv('HI_calculations/aich_experiment_hysteresis.csv', index=False)

Plot constituents

In [8]:
for experiment_name, experiment_df in experiments.items():
    for constituent in ["SSC (mg/L)", "DOC (mg/L)", "POC (mg/L)"]:
        if constituent not in experiment_df.columns:
            continue

        result = calculate_aich_HI(
            experiment_df,
            tau_col="Depth",
            constituent_col=constituent,
            storm_name=experiment_name
        )

        if result is not None:
            plot_aich_hysteresis(result, experiment_df, "Depth", constituent, out_dir="plots/aich", save=True, show=False)